In [5]:
import os
import csv
import time
import socket
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from webdriver_manager.microsoft import EdgeChromiumDriverManager
import yt_dlp
from translatepy import Translator
from langdetect import detect
from datetime import datetime
import signal
import sys
import pandas as pd
from tempfile import TemporaryDirectory
import psycopg2


In [6]:

translator = Translator()
SUPABASE_HOST = "db.vgelamgabqzpivtcqytw.supabase.co"
DB_NAME = "postgres"
USER = "postgres"
PASSWORD = "Supabase1355@"
PORT = "5432"

def connect_database():
    try:
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=USER,
            password=PASSWORD,
            host=SUPABASE_HOST,
            port=PORT
        )
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS video_info (
                id SERIAL PRIMARY KEY,
                原标题 TEXT,
                中文标题 TEXT,
                语言 TEXT,
                分辨率 TEXT,
                格式 TEXT,
                发布日期 TEXT,
                来源 TEXT,
                URL TEXT,
                CONSTRAINT unique_url UNIQUE (URL)
            )
        """)
        conn.commit()
        print("成功连接到 Supabase PostgreSQL 数据库并创建或更新 video_info 表！")
        return conn
    except Exception as e:
        print("数据库连接失败:", e)
        return None

def url_exists_in_database(connection, video_url):
    cursor = connection.cursor()
    query = "SELECT COUNT(*) FROM video_info WHERE URL = %s"
    cursor.execute(query, (video_url,))
    result = cursor.fetchone()
    cursor.close()
    return result[0] > 0

def check_network():
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=5)
        return True
    except OSError:
        return False

def get_playlist_videos(url):
    ydl_opts = {
        'quiet': True,
        'extract_flat': True,
        'playlist_items': '1-1000',
        'retries': 15,
        'fragment_retries': 15,
        'socket_timeout': 60,
    }
    for _ in range(5):
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                result = ydl.extract_info(url, download=False)
                if 'entries' in result:
                    return [(entry['url'], entry['title']) for entry in result['entries']]
                return [(url, None)]
        except Exception as e:
            print(f"获取播放列表失败: {e}，重试中...")
            time.sleep(10)
    print("获取播放列表失败，跳过...")
    return []

def get_youtube_cookies(output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    cookies_file = os.path.join(output_dir, "cookies.txt")
    try:
        with TemporaryDirectory() as temp_dir:
            options = webdriver.EdgeOptions()
            options.add_argument('--headless')
            driver = webdriver.Edge(
                service=Service(EdgeChromiumDriverManager(version="134.0.3124.85").install()),
                options=options
            )
            driver.get("https://www.youtube.com")
            time.sleep(5)
            cookies = driver.get_cookies()
            driver.quit()
        with open(cookies_file, "w") as f:
            for cookie in cookies:
                f.write(f"{cookie['name']}={cookie['value']}\n")
        print("成功生成 cookies 文件")
        return cookies_file
    except Exception as e:
        print(f"生成 cookies 失败: {str(e)}，将不使用 cookies 下载")
        return None

def download_video_and_get_info(video_url, video_title, index, cookies_file, output_dir, db_connection, max_retries=3):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    if cookies_file and not os.path.exists(cookies_file):
        print(f"未找到 {cookies_file}，将不使用 cookies 下载...")
        cookies_file = None
    
    # Check database before proceeding
    if url_exists_in_database(db_connection, video_url):
        print(f"视频 URL 已存在于数据库中，跳过下载: {video_url}")
        return None

    filename_template = os.path.join(output_dir, f"{index:03d}|%(title)s.%(ext)s")
    ydl_opts = {
        'format': 'bestvideo+bestaudio/best',
        'merge_output_format': 'mp4',
        'outtmpl': filename_template,
        'cookies': cookies_file if cookies_file else None,
        'quiet': True,
        'retries': 10,
        'fragment_retries': 10,
        'socket_timeout': 30,
        'no_check_certificate': True,
        'postprocessors': [{
            'key': 'FFmpegVideoConvertor',
            'preferedformat': 'mp4',
        }],
        'postprocessor_args': ['-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k'],
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        for attempt in range(max_retries):
            if not check_network():
                print(f"网络不可用，等待 10 秒后重试 (尝试 {attempt + 1}/{max_retries})...")
                time.sleep(10)
                continue
            try:
                print(f"开始下载视频: {video_url} (尝试 {attempt + 1}/{max_retries})")
                info = ydl.extract_info(video_url, download=False)
                original_title = video_title if video_title else info.get('title', 'N/A')
                
                part_file = filename_template.replace('.%(ext)s', '.mp4.part')
                if os.path.exists(part_file):
                    try:
                        os.remove(part_file)
                    except Exception:
                        print(f"无法删除 {part_file}，可能被占用")
                
                ydl.download([video_url])
                
                for trans_attempt in range(3):
                    try:
                        chinese_title = translator.translate(original_title, destination_language='zh').result
                        break
                    except Exception as e:
                        if trans_attempt < 2:
                            print(f"翻译失败: {e}, 重试 {trans_attempt + 1}/3...")
                            time.sleep(2)
                        else:
                            print(f"翻译失败: {e}, 使用原标题")
                            chinese_title = original_title
                
                language = detect(original_title) if original_title != 'N/A' else 'en'
                resolution = info.get('resolution', 'N/A') or f"{info.get('height', 'N/A')}p"
                format_type = 'mp4'
                upload_date = info.get('upload_date', 'N/A')
                if upload_date != 'N/A':
                    upload_date = datetime.strptime(upload_date, '%Y%m%d').strftime('%Y-%m-%d')
                uploader = info.get('uploader', 'N/A')
                
                return {
                    '原标题': original_title,
                    '中文标题': chinese_title,
                    '语言': language,
                    '分辨率': resolution,
                    '格式': format_type,
                    '发布日期': upload_date,
                    '来源': uploader,
                    'URL': video_url
                }
            except Exception as e:
                print(f"下载 {video_url} 失败: {str(e)}")
                if attempt < max_retries - 1:
                    print(f"等待 5 秒后重试...")
                    time.sleep(5)
                else:
                    print(f"达到最大重试次数 ({max_retries})，跳过视频 {video_url}")
                    return None

def insert_video_info_to_database(connection, video_info):
    cursor = connection.cursor()
    query = """
        INSERT INTO video_info (原标题, 中文标题, 语言, 分辨率, 格式, 发布日期, 来源, URL)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (URL) DO NOTHING
    """
    values = (
        video_info['原标题'],
        video_info['中文标题'],
        video_info['语言'],
        video_info['分辨率'],
        video_info['格式'],
        video_info['发布日期'],
        video_info['来源'],
        video_info['URL']
    )
    try:
        cursor.execute(query, values)
        connection.commit()
        print(f"成功写入数据库: {video_info['原标题']}")
    except Exception as e:
        connection.rollback()
        print(f"数据库写入失败: {e}")
    finally:
        cursor.close()

def append_to_csv(video_info, csv_output):
    output_dir = os.path.dirname(csv_output)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    file_exists = os.path.exists(csv_output)
    with open(csv_output, 'a', newline='', encoding='utf-8-sig') as csvfile:
        fieldnames = ['原标题', '中文标题', '语言', '分辨率', '格式', '发布日期', '来源', 'URL']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(video_info)
    
    print(f"\n更新 CSV 文件: {csv_output}")
    try:
        df = pd.read_csv(csv_output, encoding='utf-8-sig')
        print(df.to_string(index=False))
    except pd.errors.ParserError as e:
        print(f"读取 CSV 失败: {e}. 可能是字段数不匹配，请检查文件内容。")
    print("\n")

def signal_handler(sig, frame, csv_output):
    print("\n收到中断信号（Ctrl+C），程序退出...")
    sys.exit(0)

def main(playlist_url, video_output_dir="D:/Videos", csv_output="D:/Data/video_info.csv"):
    if not check_network():
        print("网络不可用，请检查你的互联网连接后重试。")
        return

    db_connection = connect_database()
    if not db_connection:
        print("无法连接数据库，程序退出。")
        return

    cookies_file = get_youtube_cookies(video_output_dir)
    video_urls = get_playlist_videos(playlist_url)

    signal.signal(signal.SIGINT, lambda sig, frame: signal_handler(sig, frame, csv_output))

    for index, (url, title) in enumerate(video_urls, start=1):
        if url_exists_in_database(db_connection, url):
            print(f"视频 URL 已存在于数据库中，跳过: {url}")
            continue

        print(f"处理视频 {index}: {url}")
        video_info = download_video_and_get_info(url, title, index, cookies_file, video_output_dir, db_connection)
        if video_info:
            append_to_csv(video_info, csv_output)
            insert_video_info_to_database(db_connection, video_info)

    db_connection.close()
    print("程序完成，所有数据库连接已关闭。")


In [8]:

if __name__ == "__main__":
    playlist_url = "https://www.youtube.com/watch?v=L5N-YVjfwMo&list=PLCye5KV8NbxR0kw1VpSIhGa5yDYPuDXof"
    video_output_dir = "D:/Videos"
    csv_output_path = "D:/Data/video_info.csv"
    main(playlist_url, video_output_dir=video_output_dir, csv_output=csv_output_path)

成功连接到 Supabase PostgreSQL 数据库并创建或更新 video_info 表！
成功生成 cookies 文件
处理视频 1: https://www.youtube.com/watch?v=L5N-YVjfwMo&list=PLCye5KV8NbxR0kw1VpSIhGa5yDYPuDXof
开始下载视频: https://www.youtube.com/watch?v=L5N-YVjfwMo&list=PLCye5KV8NbxR0kw1VpSIhGa5yDYPuDXof (尝试 1/3)


                                                                          
更新 CSV 文件: D:/Data/video_info.csv
                                                                                                原标题                                           中文标题 语言       分辨率  格式       发布日期        来源                                                                                 URL
                        Larry Nassar: US justice department to pay abuse survivors $138m | BBC News 拉里·纳萨尔（Larry Nassar）：美国司法部支付虐待幸存者1.38亿美元|BBC新闻 en  1280x720 mp4 2024-04-23  BBC News                                         https://www.youtube.com/watch?v=FZhgVT8hY84
                 Magic Johnson on the Olympics, HIV advocacy, and becoming a billionaire | BBC News                 魔术约翰逊在奥运会上，艾滋病毒倡导和成为亿万富翁|BBC新闻 en 1920x1080 mp4 2024-06-23  BBC News                                         https://www.youtube.com/watch?v=1JZZYMF0xOU
                     Paris mayor swims in Seine to prove water clean enough for Olym